# DSPy — Otimização com `SIMBA`

Neste notebook será demonstrado o uso do otimizador `SIMBA` do DSPy em um problema de **classificação binária de textos**.

Será utilizada a base **Natural Language Processing with Disaster Tweets**, disponibilizada no Kaggle. O objetivo é classificar cada tweet em uma das seguintes categorias:

* `0`: o tweet **não descreve um desastre real**;
* `1`: o tweet **descreve um desastre real**.

O experimento será dividido em duas etapas:

1. avaliar um classificador DSPy utilizando a instrução original definida na `Signature`;
2. utilizar o `SIMBA` para evoluir o programa a partir de seu desempenho sobre exemplos de treinamento.

`SIMBA` significa **Stochastic Introspective Mini-Batch Ascent**.

Em alto nível, o otimizador:

1. amostra mini-batches do conjunto de otimização;
2. executa diferentes trajetórias do programa para os mesmos exemplos;
3. identifica exemplos em que existe maior diferença de desempenho entre as trajetórias;
4. cria novos programas candidatos utilizando uma de duas estratégias:
   * adicionar uma demonstração bem-sucedida;
   * gerar uma regra de melhoria por reflexão sobre uma trajetória melhor e uma pior;
5. avalia os novos candidatos e mantém programas promissores ao longo das iterações;
6. ao final, compara candidatos selecionados sobre todo o conjunto de otimização e retorna o melhor programa.

Neste experimento, o conjunto originalmente separado em **treino** e **validação** será reunido em um único `simba_trainset`, pois o `SIMBA.compile()` recebe apenas um `trainset`. O conjunto de **teste** continuará completamente separado para a avaliação final.

A avaliação final utilizará:

* Accuracy;
* Precision;
* Recall;
* F1-score.

A métrica principal para comparar o programa original e o programa otimizado será o **F1-score**.


In [1]:
import os
from dotenv import load_dotenv  # Carrega variáveis de ambiente do arquivo .env
import dspy  # Framework para otimização de prompts com Language Models
import pandas as pd

from typing import Literal
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

from tqdm.auto import tqdm

/home/leonardo/Documentos/github/dspy_studies/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()  # Lê variáveis do arquivo .env

True

## Setup - Configuração do Modelo

Primeiro, carregamos as variáveis de ambiente, como a API key, do arquivo `.env` na raiz do projeto.

Serão utilizados dois papéis de modelo durante a otimização:

* `lm`: modelo utilizado pelo classificador e pelas trajetórias avaliadas pelo `SIMBA`;
* `prompt_lm`: modelo utilizado pelo `SIMBA` quando ele gera **regras de melhoria por reflexão**.

Neste exemplo, ambos utilizarão `openai/gpt-5-mini`.

O `SIMBA` requer a dependência opcional `numpy`. Utilizando `uv`, ela pode ser instalada com:

```bash
uv add "dspy[numpy]"
```

O NumPy já pode estar presente no ambiente por ser dependência de outros pacotes, mas o extra acima explicita a dependência requerida pelo otimizador.


In [3]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Modelo utilizado para executar o classificador
    api_key=os.getenv("OPENAI_API_KEY"),
)

# Modelo utilizado pelo SIMBA para gerar regras reflexivas de melhoria.
# Para modelos GPT-5, mantemos temperature=1.0.
prompt_lm = dspy.LM(
    "openai/gpt-5-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=1.0,
)

# Configura o modelo do classificador como LM padrão do DSPy
dspy.configure(lm=lm)


## 3. Leitura da base de dados

Será utilizada a base da competição **Natural Language Processing with Disaster Tweets**, do Kaggle.

Referência:

https://www.kaggle.com/competitions/nlp-getting-started

Para este experimento são relevantes principalmente duas colunas:

| Coluna   | Descrição                   |
| -------- | --------------------------- |
| `text`   | Texto do tweet              |
| `target` | Classe correta (`0` ou `1`) |

O problema consiste, portanto, em aprender a relação:

`text → target`


In [4]:
df = pd.read_csv('disaster_tweets.csv')
df = df.head(200)
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


## Análise da distribuição das classes

Antes da divisão dos dados, é importante verificar quantos exemplos existem de cada classe.

Além da quantidade absoluta, será analisada a proporção entre tweets classificados como `0` e `1`.

Essa análise é importante porque o **F1-score** considera conjuntamente precisão e recall e é especialmente útil quando existe algum grau de desbalanceamento entre as classes.


In [5]:
df["target"].value_counts()

target
0    103
1     97
Name: count, dtype: int64

In [6]:
df["target"].value_counts(normalize=True)

target
0    0.515
1    0.485
Name: proportion, dtype: float64

## Separação entre otimização e teste

Para manter o mesmo conjunto de teste utilizado nos notebooks anteriores, inicialmente conservaremos a divisão:

* **70% para treino**;
* **15% para validação**;
* **15% para teste**.

Entretanto, o `SIMBA` possui uma diferença importante em relação ao `MIPROv2`: seu método `compile()` **não recebe um `valset` separado**.

Por isso, depois da conversão para `dspy.Example`, os conjuntos de treino e validação serão reunidos:

```text
70% treino + 15% validação
          ↓
85% simba_trainset
```

Assim:

* `simba_trainset` contém **85% dos dados** e participa da otimização;
* `testset` contém os **15% restantes** e não participa da otimização.

O `stratify` continua sendo utilizado para manter aproximadamente a mesma proporção entre as classes `0` e `1`.

Essa estratégia também preserva o mesmo `testset` do notebook de `MIPROv2`, facilitando a comparação dos resultados finais.


In [7]:
# Primeiro separamos 70% para treino e 30% para validação + teste
df_train, df_temp = train_test_split(
    df[["text", "target"]],
    test_size=0.30,
    random_state=42,
    stratify=df["target"],
)

# Divide os 30% restantes igualmente: 15% validação e 15% teste
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp["target"],
)

print(f"Treino:     {len(df_train)} exemplos")
print(f"Validação:  {len(df_val)} exemplos")
print(f"Teste:      {len(df_test)} exemplos")

Treino:     140 exemplos
Validação:  30 exemplos
Teste:      30 exemplos


## Conversão para `dspy.Example`

O DSPy representa exemplos de treino e teste por meio da classe `dspy.Example`.

Neste problema, cada exemplo possui dois campos:

* `text`: entrada fornecida ao modelo;
* `target`: resposta esperada.

A chamada:

`with_inputs("text")`

informa explicitamente ao DSPy que `text` deve ser utilizado como entrada do programa.

Consequentemente, `target` passa a ser tratado como o **label**, ou seja, a resposta esperada para aquele exemplo.

Conceitualmente, cada registro passa a ter a seguinte estrutura:

`entrada: text → saída esperada: target`

In [8]:
def dataframe_para_dspy(dataframe):
    exemplos = []

    for _, row in dataframe.iterrows():
        exemplo = dspy.Example(
            text=row["text"],
            target=int(row["target"]),
        ).with_inputs("text")

        exemplos.append(exemplo)

    return exemplos

In [9]:
trainset = dataframe_para_dspy(df_train)
valset = dataframe_para_dspy(df_val)
testset = dataframe_para_dspy(df_test)

# O SIMBA não possui um valset separado no compile().
# Reunimos treino + validação para formar o conjunto efetivamente
# utilizado durante a otimização.
simba_trainset = trainset + valset

print(f"Trainset original: {len(trainset)}")
print(f"Valset original:   {len(valset)}")
print(f"SIMBA trainset:    {len(simba_trainset)}")
print(f"Testset:           {len(testset)}")


Trainset original: 140
Valset original:   30
SIMBA trainset:    170
Testset:           30


In [10]:
trainset[0]

Example({'text': '13,000 people receive #wildfires evacuation orders in California ', 'target': 1}) (input_keys={'text'})

## Definição da tarefa com uma `Signature`

No DSPy, uma `Signature` descreve declarativamente a tarefa que será executada pelo modelo.

A `ClassificarTweet` possui:

* um `InputField` chamado `text`, contendo o tweet;
* um `OutputField` chamado `target`, contendo a classificação.

O tipo:

`Literal[0, 1]`

restringe a resposta esperada às duas classes válidas do problema.

Dessa forma, a Signature define claramente o contrato:

`texto do tweet → 0 ou 1`


In [11]:
class ClassificarTweet(dspy.Signature):
    """
    Classifique o tweet em uma das duas classes possíveis.
    """

    text: str = dspy.InputField(
        desc="Tweet a ser analisado."
    )

    target: Literal[0, 1] = dspy.OutputField(
        desc="Classe prevista."
    )

In [12]:
classificador_base = dspy.Predict(ClassificarTweet)

In [13]:
def avaliar_classificador(programa, dataset, descricao="Avaliando"):
    """
    Executa um programa DSPy sobre um dataset e calcula
    métricas globais de classificação.
    """

    y_true = []
    y_pred = []

    for exemplo in tqdm(dataset, desc=descricao):

        # Executa o programa utilizando somente os campos
        # marcados como entrada pelo with_inputs(...)
        predicao = programa(**exemplo.inputs())

        # Label verdadeiro
        y_true.append(int(exemplo.target))

        # Label previsto pelo DSPy
        y_pred.append(int(predicao.target))

    # Calcula as métricas sobre todo o conjunto
    resultado = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }

    return resultado

In [14]:
resultado_base = avaliar_classificador(
    classificador_base,
    testset,
    descricao="Baseline zero-shot",
)

Baseline zero-shot: 100%|█████████| 30/30 [00:01<00:00, 15.66it/s]


## Avaliação do baseline

O classificador inicial será executado sobre todos os exemplos do conjunto de teste.

Para cada exemplo:

1. o campo `text` é enviado ao programa;
2. o programa produz uma previsão para `target`;
3. a previsão é comparada com o `target` verdadeiro.

Ao final são calculadas as métricas globais de classificação.

Esse resultado será considerado o desempenho **antes da otimização**.


In [15]:
print(f"F1 baseline: {resultado_base['f1']:.4f}")

F1 baseline: 0.8966


In [16]:
print(f"Accuracy:  {resultado_base['accuracy']:.4f}")
print(f"Precision: {resultado_base['precision']:.4f}")
print(f"Recall:    {resultado_base['recall']:.4f}")
print(f"F1:        {resultado_base['f1']:.4f}")

Accuracy:  0.9000
Precision: 0.9286
Recall:    0.8667
F1:        0.8966


## Métrica utilizada durante a otimização

O `SIMBA` precisa de uma métrica aplicada a cada exemplo para atribuir um **score** às diferentes trajetórias e aos programas candidatos.

Neste problema será utilizada uma métrica simples de **acerto da classe**:

```python
def metrica_simba(example, prediction):
    esperado = int(example.target)
    previsto = int(prediction.target)

    return float(esperado == previsto)
```

A métrica retorna:

* `1.0` quando a classe prevista coincide com a classe esperada;
* `0.0` quando a classificação está incorreta.

Durante cada etapa, o SIMBA executa múltiplas trajetórias sobre exemplos de um mini-batch. As diferenças de score ajudam o otimizador a identificar casos úteis para gerar novas demonstrações ou regras reflexivas.

O **F1-score global** continuará sendo calculado separadamente sobre todo o conjunto de teste. Isso é importante porque o F1 é uma métrica agregada e não pode ser calculado corretamente a partir de um único exemplo isolado.


In [17]:
def metrica_simba(example, prediction):
    """
    Métrica utilizada internamente pelo SIMBA.

    Retorna 1.0 quando a classe prevista é igual à classe esperada
    e 0.0 caso contrário.
    """
    esperado = int(example.target)
    previsto = int(prediction.target)

    return float(esperado == previsto)


## Otimização com `SIMBA`

O `SIMBA` (**Stochastic Introspective Mini-Batch Ascent**) é um otimizador que melhora o programa iterativamente a partir da análise de suas próprias execuções.

Para cada etapa de otimização, ele trabalha aproximadamente da seguinte forma:

```text
mini-batch do simba_trainset
            ↓
múltiplas trajetórias por exemplo
            ↓
cálculo da métrica
            ↓
identificação de exemplos com resultados contrastantes
            ↓
┌──────────────────────────┬─────────────────────────────┐
│ adicionar demonstração   │ gerar regra por reflexão   │
│ bem-sucedida             │ melhor vs. pior trajetória │
└──────────────────────────┴─────────────────────────────┘
            ↓
novos programas candidatos
            ↓
avaliação no mini-batch
            ↓
pool de candidatos
```

Ao final das etapas, alguns programas representativos da trajetória de otimização são avaliados sobre todo o `simba_trainset`, e o melhor é retornado.

Neste notebook usaremos um orçamento menor que os valores padrão para tornar o experimento mais prático com chamadas reais à API:

* `bsize=16`: 16 exemplos em cada mini-batch;
* `num_candidates=3`: três amostragens/candidatos principais por etapa;
* `max_steps=3`: três etapas de otimização;
* `max_demos=4`: no máximo quatro demonstrações armazenadas por preditor.

Para uma busca mais extensa, esses valores podem ser aumentados. Os padrões atuais do DSPy são `bsize=32`, `num_candidates=6`, `max_steps=8` e `max_demos=4`.


In [18]:
BSIZE = 16
NUM_CANDIDATES = 3
MAX_STEPS = 3
MAX_DEMOS = 4

optimizer = dspy.SIMBA(
    metric=metrica_simba,                 # Métrica aplicada a cada previsão
    bsize=BSIZE,                           # Quantidade de exemplos em cada mini-batch
    num_candidates=NUM_CANDIDATES,         # Quantidade de amostragens/candidatos por etapa
    max_steps=MAX_STEPS,                   # Número de etapas de otimização
    max_demos=MAX_DEMOS,                   # Máximo de demonstrações por preditor
    prompt_model=prompt_lm,                # LM usado para gerar regras reflexivas
    num_threads=4,                         # Paralelismo das execuções
    temperature_for_sampling=0.2,          # Exploração ao escolher programas para trajetórias
    temperature_for_candidates=0.2,        # Exploração ao escolher programas-base de candidatos
)


## Compilação do programa

A compilação do `SIMBA` recebe:

* `student`: o programa que será otimizado;
* `trainset`: os exemplos utilizados durante todo o processo de otimização;
* `seed`: semente utilizada nas escolhas aleatórias internas.

```python
classificador_otimizado = optimizer.compile(
    student=classificador_base,
    trainset=simba_trainset,
    seed=42,
)
```

Diferentemente do `MIPROv2`, não existe um parâmetro `valset` em `SIMBA.compile()`.

Também é importante garantir que:

```python
len(simba_trainset) >= BSIZE
```

pois o otimizador exige que o conjunto de treinamento tenha pelo menos a quantidade de exemplos definida em `bsize`.


In [19]:
classificador_otimizado = optimizer.compile(
    student=classificador_base,
    trainset=simba_trainset,
    seed=42,
)


2026/09/08 07:16:05 INFO dspy.teleprompt.simba: Starting batch 1 of 3.
2026/09/08 07:16:05 INFO dspy.teleprompt.simba: Sampling program trajectories on 16 examples x 3 samples.


Processed 48 / 48 examples: 100%|█| 48/48 [01:36<00:00,  2.01s/it]

2026/09/08 07:17:41 INFO dspy.teleprompt.simba: Batch 1: Baseline mini-batch score: 0.8125

2026/09/08 07:17:41 INFO dspy.teleprompt.simba: Batch 1: Processing bucket #1, with max score 1.0, max-to-min gap 1.0, and max-to-avg gap 0.6666666666666667.
2026/09/08 07:17:41 INFO dspy.teleprompt.simba: Batch 1: Invoking strategy: append_a_demo_
2026/09/08 07:17:41 INFO dspy.teleprompt.simba_utils: Added 1 demos (one each) across all predictors.
2026/09/08 07:17:41 INFO dspy.teleprompt.simba: 

2026/09/08 07:17:41 INFO dspy.teleprompt.simba: Batch 1: Processing bucket #2, with max score 1.0, max-to-min gap 1.0, and max-to-avg gap 0.6666666666666667.
2026/09/08 07:17:41 INFO dspy.teleprompt.simba: Batch 1: Invoking strategy: append_a_rule
2026/09/08 07:17:41 WARNING dspy.predict.predict: Type mismatch for field 'worse_reward_value': expected float based on given Signature, but the provided value is incompatible: 0.0.
2026/09/08 07:17:41 WARNING dspy.predict.predict: Type mismatch for field 'be

2026/09/08 07:17:56 INFO dspy.teleprompt.simba_utils: Advice for self: When presented with a tweet text, first normalize it (lowercase, remove/strip URLs and surrounding whitespace, collapse repeating spaces). Then apply these concrete checks in order: (1) Keyword detection: if the cleaned text contains any event keywords or multiword phrases such as: shooting, shot, stabbing, explosion, bomb, fire, crash, accident, plane crash, airplane accident, train crash, collision, killed, injured, death, dead, hostage, attack, mass shooting, massacre, stabbing, car crash, tornado, earthquake, flood, hurricane — predict target = 1. Match whole words and common variants (e.g., 'shot' vs 'shooting'). (2) Handle conjunctions/uncertainty: words like 'or', 'maybe', 'possibly' do NOT negate an event mention; still predict 1 if an event keyword appears (the presence of 'or' in "shooting or the airplane accident" should still be positive). (3) Negation handling: if an explicit negation appears within ~3 

Processed 64 / 64 examples: 100%|█| 64/64 [02:11<00:00,  2.05s/it]

...

2026/09/08 07:26:09 INFO dspy.teleprompt.simba: Batch 3: Baseline mini-batch score: 0.8958333333333334

2026/09/08 07:26:09 INFO dspy.teleprompt.simba: Batch 3: Processing bucket #1, with max score 1.0, max-to-min gap 1.0, and max-to-avg gap 0.6666666666666667.
2026/09/08 07:26:09 INFO dspy.teleprompt.simba: Batch 3: Invoking strategy: append_a_demo_
2026/09/08 07:26:09 INFO dspy.teleprompt.simba_utils: Added 1 demos (one each) across all predictors.
2026/09/08 07:26:09 INFO dspy.teleprompt.simba: 

2026/09/08 07:26:09 INFO dspy.teleprompt.simba: Batch 3: Processing bucket #2, with max score 1.0, max-to-min gap 0.0, and max-to-avg gap 0.0.
2026/09/08 07:26:09 INFO dspy.teleprompt.simba: Batch 3: Invoking strategy: append_a_rule, having dropped 1 demos per predictor
2026/09/08 07:26:09 INFO dspy.teleprompt.simba_utils: Skipping rule generation as good score 1.0 is at or below the 10th percentile *or* bad score 1.0 is at or above the 90th percentile.
2026/09/08 07:26:09 INFO dspy.telepro


Processed 64 / 64 examples: 100%|█| 64/64 [02:21<00:00,  2.22s/it]

2026/09/08 07:28:31 INFO dspy.teleprompt.simba: Scores after 3 batches: [0.9375, 0.5625, 0.9375, 0.9375], Best: 0.9375

2026/09/08 07:28:31 INFO dspy.teleprompt.simba: VALIDATION: Evaluating 4 programs on the full trainset.



Processed 680 / 680 examples: 100%|█| 680/680 [18:05<00:00,  1.60s

2026/09/08 07:46:37 INFO dspy.teleprompt.simba: Final trainset scores: [0.8411764705882353, 0.8588235294117647, 0.9352941176470588, 0.9294117647058824], Best: 0.9352941176470588 (at index 2)





## Inspeção do resultado da otimização

O programa retornado pelo `SIMBA` recebe dois atributos úteis para inspeção:

* `candidate_programs`: programas candidatos finais, acompanhados de seus scores;
* `trial_logs`: informações registradas durante as etapas da otimização.

Além disso, devemos inspecionar duas partes do preditor otimizado:

1. **instruções**;
2. **demonstrações few-shot**.

Isso é particularmente importante no SIMBA porque ele possui duas estratégias de melhoria:

* **regra reflexiva**: uma recomendação gerada pelo modelo é acrescentada às instruções existentes do preditor;
* **demonstração**: uma trajetória bem-sucedida pode ser adicionada como exemplo few-shot.

Portanto, a instrução final pode permanecer igual à original e ainda assim o programa ser diferente por causa das demonstrações. Da mesma forma, quando uma regra é selecionada, normalmente veremos texto adicional anexado à instrução original.


In [20]:
print("=== INSTRUÇÃO ORIGINAL ===")
print(classificador_base.signature.instructions)

print("\n=== INSTRUÇÃO APÓS O SIMBA ===")
print(classificador_otimizado.signature.instructions)

print("\n=== DEMONSTRAÇÕES FEW-SHOT SELECIONADAS ===")
print(f"Quantidade de demos: {len(classificador_otimizado.demos)}")

for i, demo in enumerate(classificador_otimizado.demos, start=1):
    print(f"\nDemo {i}:")
    print(demo)


=== INSTRUÇÃO ORIGINAL ===
Classifique o tweet em uma das duas classes possíveis.

=== INSTRUÇÃO APÓS O SIMBA ===
Classifique o tweet em uma das duas classes possíveis.

=== DEMONSTRAÇÕES FEW-SHOT SELECIONADAS ===
Quantidade de demos: 2

Demo 1:
Example({'augmented': True, 'text': 'Could a drone cause an airplane accident? Pilots worried about use of drones esp. in close vicinity of airports http://t.co/kz35rGngJF #', 'target': 1}) (input_keys=None)

Demo 2:
Example({'augmented': True, 'text': 'airplane crashes on house in Colombia 12 people die in accident https://t.co/ZhJlfLBHZL', 'target': 1}) (input_keys=None)


In [21]:
print(f"Quantidade de candidatos finais: {len(classificador_otimizado.candidate_programs)}")
print(f"Quantidade de etapas registradas: {len(classificador_otimizado.trial_logs)}")

print("\n=== SCORES DOS CANDIDATOS FINAIS ===")
for i, candidato in enumerate(classificador_otimizado.candidate_programs, start=1):
    print(f"Candidato {i}: {candidato['score']:.4f}")

print("\n=== TRIAL LOGS ===")
print(classificador_otimizado.trial_logs)


Quantidade de candidatos finais: 4
Quantidade de etapas registradas: 3

=== SCORES DOS CANDIDATOS FINAIS ===
Candidato 1: 0.9353
Candidato 2: 0.9294
Candidato 3: 0.8588
Candidato 4: 0.8412

=== TRIAL LOGS ===
{0: {'train_score': 0.8588235294117647}, 1: {'train_score': 0.9352941176470588}, 2: {'train_score': 0.9294117647058824}}


In [22]:
historico_simba = pd.DataFrame(
    [
        {
            "candidato": i,
            "score_treino": candidato["score"],
        }
        for i, candidato in enumerate(
            classificador_otimizado.candidate_programs,
            start=1,
        )
    ]
).sort_values(
    "score_treino",
    ascending=False,
)

historico_simba


,candidato,score_treino
0,1,0.935294
1,2,0.929412
2,3,0.858824
3,4,0.841176


In [23]:
resultado_otimizado = avaliar_classificador(
    classificador_otimizado,
    testset,
    descricao="SIMBA",
)


SIMBA: 100%|██████████████████████| 30/30 [03:11<00:00,  6.39s/it]


## Avaliação após a aplicação do `SIMBA`

O programa otimizado pelo `SIMBA` será avaliado utilizando **exatamente o mesmo conjunto de teste utilizado pelo baseline**.

Isso permite comparar:

* o classificador utilizando apenas a instrução original;
* o classificador resultante do processo de otimização do SIMBA, que pode conter regras adicionais, demonstrações few-shot ou ambos.

O `testset` não participou da otimização.

Durante a otimização:

* o `simba_trainset` foi amostrado em mini-batches;
* diferentes trajetórias foram executadas para os mesmos exemplos;
* a métrica individual permitiu comparar essas trajetórias;
* o SIMBA criou candidatos adicionando demonstrações bem-sucedidas ou regras reflexivas;
* ao final, candidatos selecionados foram avaliados sobre todo o `simba_trainset`.

Serão calculados novamente:

* Accuracy;
* Precision;
* Recall;
* F1-score.

O **F1-score global do conjunto de teste** continuará sendo a métrica principal para a comparação final.


In [24]:
comparacao = pd.DataFrame(
    {
        "Modelo": [
            "Baseline (instrução original)",
            "SIMBA",
        ],
        "Accuracy": [
            resultado_base["accuracy"],
            resultado_otimizado["accuracy"],
        ],
        "Precision": [
            resultado_base["precision"],
            resultado_otimizado["precision"],
        ],
        "Recall": [
            resultado_base["recall"],
            resultado_otimizado["recall"],
        ],
        "F1": [
            resultado_base["f1"],
            resultado_otimizado["f1"],
        ],
    }
)

comparacao


,Modelo,Accuracy,Precision,Recall,F1
0,Baseline (instrução original),0.900000,0.928571,0.866667,0.896552
1,SIMBA,0.866667,0.789474,1.000000,0.882353


In [25]:
print("BASELINE")
print(
    classification_report(
        resultado_base["y_true"],
        resultado_base["y_pred"],
        digits=4,
    )
)

BASELINE
              precision    recall  f1-score   support

           0     0.8750    0.9333    0.9032        15
           1     0.9286    0.8667    0.8966        15

    accuracy                         0.9000        30
   macro avg     0.9018    0.9000    0.8999        30
weighted avg     0.9018    0.9000    0.8999        30



In [26]:
print(
    f"SIMBA (bsize={BSIZE}, "
    f"num_candidates={NUM_CANDIDATES}, "
    f"max_steps={MAX_STEPS})"
)

print(
    classification_report(
        resultado_otimizado["y_true"],
        resultado_otimizado["y_pred"],
        digits=4,
    )
)


SIMBA (bsize=16, num_candidates=3, max_steps=3)
              precision    recall  f1-score   support

           0     1.0000    0.7333    0.8462        15
           1     0.7895    1.0000    0.8824        15

    accuracy                         0.8667        30
   macro avg     0.8947    0.8667    0.8643        30
weighted avg     0.8947    0.8667    0.8643        30



## Salvando o programa otimizado pelo `SIMBA`

Neste experimento, o programa possui uma arquitetura simples baseada em `dspy.Predict`.

O `SIMBA` pode alterar o estado do preditor ao:

* acrescentar **regras** às instruções;
* adicionar **demonstrações few-shot**.

Por isso, o **State-only Saving** em JSON é adequado:

```python
classificador_otimizado.save("SIMBA.json")
```

Esse arquivo salva o estado otimizado necessário para reutilizar o programa, mas não a definição Python completa da arquitetura.

Para carregar posteriormente, recriamos a mesma arquitetura (`dspy.Predict(ClassificarTweet)`) e aplicamos `.load()`.

Os atributos de análise da busca, como `candidate_programs` e `trial_logs`, não são necessários para executar o classificador depois de carregado.


In [27]:
classificador_otimizado.save("SIMBA.json")


In [28]:
# Recria a mesma arquitetura do programa
classificador_carregado = dspy.Predict(ClassificarTweet)

# Carrega o estado otimizado pelo SIMBA
classificador_carregado.load("SIMBA.json")

classificador_carregado


Predict(StringSignature(text -> target
    instructions='Classifique o tweet em uma das duas classes possíveis.'
    text = Field(annotation=str required=True json_schema_extra={'desc': 'Tweet a ser analisado.', '__dspy_field_type': 'input', 'prefix': 'Text:'})
    target = Field(annotation=Literal[0, 1] required=True json_schema_extra={'desc': 'Classe prevista.', '__dspy_field_type': 'output', 'prefix': 'Target:'})
))

In [29]:
tweet = "My phone battery died right before the meeting, what a disaster!"

predicao = classificador_carregado(
    text=tweet
)

print(predicao)

Prediction(
    target=0
)


In [30]:
# Mostra última chamada ao modelo (n=1 significa 1 última chamada)
dspy.inspect_history(n=1)





[2026-09-08T07:49:54.950652]

System message:

Your input fields are:
1. `text` (str): Tweet a ser analisado.
Your output fields are:
1. `target` (Literal[0, 1]): Classe prevista.
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## text ## ]]
{text}

Outputs will be a JSON object with the following fields.

{
  "target": "{target}        # note: the value you produce must exactly match (no extra characters) one of: 0; 1"
}
In adhering to this structure, your objective is: 
        Classifique o tweet em uma das duas classes possíveis.


User message:

[[ ## text ## ]]
Could a drone cause an airplane accident? Pilots worried about use of drones esp. in close vicinity of airports http://t.co/kz35rGngJF #


Assistant message:

{
  "target": 1
}


User message:

[[ ## text ## ]]
airplane crashes on house in Colombia 12 people die in accident https://t.co/ZhJlfLBHZL


Assistant message:

{
  "